# P03 — Per-seed severity correlation evaluation

Computes the per-seed severity Pearson correlation for the CPA n=100 sweep and the matched-n GEARS audit.

**Manuscript reference:** Methods §2.4, §2.5; Results §3.1.

The severity Pearson is the per-seed correlation between predicted perturbation magnitude (the model's mean absolute delta from the unperturbed control baseline, averaged across the evaluated gene set) and the per-perturbation leverage score, across the 30 held-out perturbations.

**Inputs (one h5ad per seed):**
`data/predictions/severity_details/{model}_{cell_type}_{split_type}_seed{seed}.severity.h5ad`

Each h5ad has the following columns in `.obs`:
- `predicted_mean_abs_delta` — per-perturbation predicted magnitude
- `leverage_score` — per-perturbation severity reference (from the Replogle 2022 supplement)
- `perturbation_target` — gene symbol (used for the LOO mode-driver in P04)

These h5ads are produced by P01 (CPA) and P02 (GEARS). If `data/predictions/severity_details/` does not exist, this notebook will fall back to documenting expected behaviour and exit without writing output.

**Outputs (per-seed):**
- `data/eval/per_seed_severity.csv` — one row per (model, cell_type, split_type, seed) with `severity_pearson_raw`.

To copy these outputs into `precomputed/eval/` as the shipped reproducibility layer, see the regeneration instructions in `precomputed/README.md`.


In [1]:
import sys
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
from scipy.stats import pearsonr

sys.path.insert(0, str(Path.cwd()))
from perturb_style import DATA_DIR, PRECOMPUTED_DIR


## Configuration

In [2]:
PREDICTIONS_DIR = DATA_DIR / "predictions" / "severity_details"
EVAL_OUT_DIR = DATA_DIR / "eval"
EVAL_OUT_DIR.mkdir(exist_ok=True, parents=True)

# CPA sweep: 4 conditions x 100 seeds. GEARS matched-n: 4 conditions x 7 seeds.
CPA_SEEDS = list(range(1, 101))
GEARS_SEEDS = list(range(1, 8))
CELL_TYPES = ["K562", "RPE1"]
SPLITS = ["random", "stratified"]


## Helpers

In [3]:
def severity_h5ad_path(model: str, cell_type: str, split_type: str, seed: int) -> Path:
    """Standardised on-disk location for per-seed severity detail h5ads."""
    fname = f"{model}_{cell_type}_{split_type}_seed{seed}.severity.h5ad"
    return PREDICTIONS_DIR / fname

def evaluate_one_seed(model: str, cell_type: str, split_type: str, seed: int) -> dict | None:
    """Read a per-seed severity detail h5ad and compute the Pearson correlation.

    Returns a record dict, or None if the file is missing (skip-and-log)."""
    import anndata as ad
    path = severity_h5ad_path(model, cell_type, split_type, seed)
    if not path.exists():
        return None
    obs = ad.read_h5ad(path).obs
    mag = obs["predicted_mean_abs_delta"].to_numpy(float)
    lev = obs["leverage_score"].to_numpy(float)
    r = float(pearsonr(mag, lev)[0])
    return {
        "model_id": model,
        "cell_type": cell_type,
        "split_type": split_type,
        "seed": seed,
        "severity_pearson_raw": round(r, 4),
        "n_perturbations": int(len(mag)),
    }


## Run the evaluation

If `data/predictions/severity_details/` does not exist (the public user has not yet generated predictions), this cell prints a helpful message and exits without writing output. The figure notebooks (01–05) operate from `precomputed/eval/` and do not depend on this notebook in that path.

To populate `data/predictions/severity_details/`, run P01 (CPA, ~13 hours) and P02 (GEARS, ~2 hours), or `./run_all.sh --demo` for a 2-seed verification run.

In [4]:
if not PREDICTIONS_DIR.exists() or not any(PREDICTIONS_DIR.iterdir()):
    print(f"Predictions directory not found or empty: {PREDICTIONS_DIR}")
    print("This notebook produces data/eval/per_seed_severity.csv from per-seed")
    print("prediction h5ads written by P01 (CPA) and P02 (GEARS).")
    print()
    print("To produce predictions:")
    print("  ./run_all.sh --demo        # 2-seed verification (~30 min)")
    print("  ./run_all.sh --train-full  # full n=100 CPA + matched-n GEARS (~13 hours)")
    print()
    print("To skip prediction-generation and use the shipped precomputed CSVs:")
    print("  ./run_all.sh --figures     # reads precomputed/eval/ directly")
    print()
    print(f"Existing precomputed CSV: {PRECOMPUTED_DIR / 'eval' / 'stage5_comparison_n100.csv'}")
    print("Exiting without writing output.")
else:
    rows = []
    missing = []
    for model, seeds in [("CPA", CPA_SEEDS), ("GEARS", GEARS_SEEDS)]:
        for cell, split, seed in product(CELL_TYPES, SPLITS, seeds):
            r = evaluate_one_seed(model, cell, split, seed)
            if r is None:
                missing.append(f"{model} {cell} {split} seed{seed}")
                continue
            rows.append(r)

    per_seed = pd.DataFrame(rows)
    out_path = EVAL_OUT_DIR / "per_seed_severity.csv"
    per_seed.to_csv(out_path, index=False)

    print(f"Wrote {out_path} ({len(per_seed)} rows)")
    print(f"Missing per-seed h5ads: {len(missing)}")
    if missing[:5]:
        for m in missing[:5]:
            print(f"  {m}")
        if len(missing) > 5:
            print(f"  ... and {len(missing) - 5} more")

    print("\n=== Counts by (model, cell_type, split_type) ===")
    print(per_seed.groupby(["model_id", "cell_type", "split_type"]).size().to_string())


Predictions directory not found or empty: data/predictions/severity_details
This notebook produces data/eval/per_seed_severity.csv from per-seed
prediction h5ads written by P01 (CPA) and P02 (GEARS).

To produce predictions:
  ./run_all.sh --demo        # 2-seed verification (~30 min)
  ./run_all.sh --train-full  # full n=100 CPA + matched-n GEARS (~13 hours)

To skip prediction-generation and use the shipped precomputed CSVs:
  ./run_all.sh --figures     # reads precomputed/eval/ directly

Existing precomputed CSV: precomputed/eval/stage5_comparison_n100.csv
Exiting without writing output.
